In [1]:
import os
import pandas as pd
import geopandas as gpd

In [3]:
import sys
sys.path.append('../')

In [4]:
report_path = r'D:\WORK\wildfire\Interagency-Tracking-System\its\ITSGDB_backup\V2.0.2\reports.gdb'

In [5]:
old_report_path = r'D:\WORK\wildfire\Interagency-Tracking-System\its\ITSGDB_backup\V2.0.1\reports.gdb'

In [6]:
gpd.list_layers(old_report_path)

,name,geometry_type
0,activity_report20250814,Point
1,footprint2021,MultiPolygon
2,footprint2022,MultiPolygon
3,footprint2023,MultiPolygon
4,footprint2024,MultiPolygon


In [19]:
activity_out_path = r'D:\WORK\wildfire\Interagency-Tracking-System\its\ITSGDB_backup\V2.1upload\activity_stats.xlsx'
footprint_out_path = r'D:\WORK\wildfire\Interagency-Tracking-System\its\ITSGDB_backup\V2.1upload\footprint_stats.xlsx'

In [4]:
upload_path = r"D:\WORK\wildfire\Interagency-Tracking-System\its\ITSGDB_backup\V2.1upload"

In [6]:
activity_gdf = gpd.read_file(os.path.join(upload_path, 'ITSV2.1_Activity.gdb'), 
                             driver='OpenFileGDB',
                             layer='ITSV2_1_Activity')

footprint_gdf = gpd.read_file(os.path.join(upload_path, 'ITSV2.1_Footprint_Polygons.gdb'), 
                             driver='OpenFileGDB',
                             layer='ITSV2_1_Footprint_Polygons')

C:\Users\sky\.conda\envs\its_recode\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: driver OpenFileGDB does not support open option DRIVER
  return ogr_read(
C:\Users\sky\.conda\envs\its_recode\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  return ogr_read(


In [7]:
old_upload_path = r"D:\WORK\wildfire\Interagency-Tracking-System\its\ITSGDB_backup\V2.0Fall_upload"

In [9]:
gpd.list_layers(os.path.join(old_upload_path, 'ITS_V2_0_Activity_webmap.gdb'))

,name,geometry_type
0,ITS_V2_0_Activity,Point


In [10]:
gpd.list_layers(os.path.join(old_upload_path, 'ITS_V2_0_Footprint_Polygons.gdb'))

,name,geometry_type
0,ITS_V2_0_Footprint_Polygons_gdb,MultiPolygon


In [11]:
old_activity_gdf = gpd.read_file(os.path.join(old_upload_path, 'ITS_V2_0_Activity_webmap.gdb'), 
                             driver='OpenFileGDB',
                             layer='ITS_V2_0_Activity')
old_footprint_gdf = gpd.read_file(os.path.join(old_upload_path, 'ITS_V2_0_Footprint_Polygons.gdb'), 
                             driver='OpenFileGDB',
                             layer='ITS_V2_0_Footprint_Polygons_gdb')

C:\Users\sky\.conda\envs\its_recode\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: driver OpenFileGDB does not support open option DRIVER
  return ogr_read(
C:\Users\sky\.conda\envs\its_recode\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: driver OpenFileGDB does not support open option DRIVER
  return ogr_read(


In [12]:
def get_report_stats(df, col_name):
    # activity acres by input column name
    pivot_table = df.pivot_table(
        index=col_name,
        columns="Year_txt",
        values="ACTIVITY_QUANTITY",
        aggfunc="sum",
        fill_value=0
    )
    
    pivot_table.loc['TOTAL',:] = pivot_table.sum()
    return pivot_table

In [80]:
# add CORE_CRITERIA to append and report for database

# maybe use integer for CORE_CRITERIA

array(['VALID', 'FIXED'], dtype=object)

In [13]:
activity_cols = ['ADMINISTERING_ORG', 'PRIMARY_OWNERSHIP_GROUP', 'REGION', 'ACTIVITY_CAT', 'BROAD_VEGETATION_TYPE']
footprint_cols = ['PRIMARY_OWNERSHIP_GROUP', 'REGION', 'BROAD_VEGETATION_TYPE']


In [18]:
def generate_excel(gdf, old_gdf, stats_cols, diff_cols, output_path, report_sheet_name):
    # init writer
    writer = pd.ExcelWriter(output_path, engine="xlsxwriter") 
    workbook = writer.book
    # create format oject that convert excel number format to comma seperated 2 decimal precision
    cellFormat = workbook.add_format({'num_format': '#,##.00'})
    # save original report to excel
    gdf.drop(columns=['geometry']).to_excel(writer, sheet_name=report_sheet_name) 
    
    
    for col in stats_cols:
        # save current summary statistic pivot table to sheet
        stats_df = get_report_stats(gdf, col)
        stats_df.to_excel(writer, sheet_name=col +'_stats')
        
        # format float to comma seperate with 2 decimal points
        worksheet = writer.sheets[col +'_stats']
        worksheet.set_column('A:{}'.format(chr(65+stats_df.shape[1])), stats_df.shape[0]+1, cellFormat)

        
    # create stat differences between versions
    for col in diff_cols:
        new_stats_df = get_report_stats(gdf, col)
        old_stats_df = get_report_stats(old_gdf, col)
        out_df = new_stats_df[old_stats_df.columns] - old_stats_df
        out_df.to_excel(writer, sheet_name=col +'_diff')
        
        # format float to comma seperate with 2 decimal points
        worksheet = writer.sheets[col +'_diff']
        worksheet.set_column('A:{}'.format(chr(65+out_df.shape[1])), out_df.shape[0]+1, cellFormat)
        
    writer.close()

In [29]:
activity_gdf = activity_gdf[activity_gdf.Year_txt.isin(['2021', '2022', '2023', '2024', '2025'])]

In [30]:
generate_excel(activity_gdf, old_activity_gdf, activity_cols, activity_cols, activity_out_path, 'activity_report')

In [21]:
generate_excel(footprint_gdf, old_footprint_gdf, footprint_cols, footprint_cols, footprint_out_path, 'footprint_report')

In [ ]:
diff_cols = ['ACTIVITY_CAT', 'ACTIVITY_DESCRIPTION', 'ADMINISTERING_ORG', 'Year_txt']

In [ ]:
def generate_full_comparison_csv(gdf, old_gdf, diff_cols, output_path):
    # aggregate ACTIVITY_QUANTITY based on list of stat columns
    # default to ['ACTIVITY_CAT', 'ACTIVITY_DESCRIPTION', 'ADMINISTERING_ORG', 'Year_txt']
    subset_cols = diff_cols + ['ACTIVITY_QUANTITY']
    gdf = gdf[subset_cols]
    old_gdf = old_gdf[subset_cols]
    agg_gdf = gdf.groupby(diff_cols).agg({'ACTIVITY_QUANTITY':'sum'}).reset_index()
    agg_gdf = agg_gdf.sort_values(
        by=diff_cols,
        ascending=[True, True, True]
    )
    agg_old_gdf = old_gdf.groupby(diff_cols).agg({'ACTIVITY_QUANTITY':'sum'}).reset_index()
    agg_old_gdf = agg_old_gdf.sort_values(
        by=diff_cols,
        ascending=[True, True, True]
    )

    # outer join old and new aggregated df
    comparison_df = agg_gdf.merge(
        agg_old_gdf,
        on=diff_cols,
        how='outer'
    )
    
    # fill missing values with zeros
    comparison_df['ACTIVITY_QUANTITY_x'] = comparison_df['ACTIVITY_QUANTITY_x'].fillna(0)
    comparison_df['ACTIVITY_QUANTITY_y'] = comparison_df['ACTIVITY_QUANTITY_y'].fillna(0)

    # create a final difference column to show the difference in area 
    comparison_df['ACTIVITY_QUANTITY_diff'] = comparison_df['ACTIVITY_QUANTITY_x'] - comparison_df['ACTIVITY_QUANTITY_y']
    comparison_df['ACTIVITY_QUANTITY_abs'] = comparison_df['ACTIVITY_QUANTITY_diff'].abs()
    comparison_df.sort_values(by='ACTIVITY_QUANTITY_diff', ascending=False, inplace=True)

    # save to file
    comparison_df.to_csv(output_path)

    return comparison_df

In [ ]:
generate_full_comparison_csv(activity_gdf, old_activity_gdf, diff_cols, 'ITS_activity_v2.0_vs_v2.1_comparison.csv')